# Analysis of n3r6 pipeline run
Initial analysis of antibody panel 2 data

In [26]:
import pathlib as Path
import pandas as pd
import os
import string

import glob

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Metadata
Alena aggregated all of the cell culture group's metadata information: iNDI_Plate_ID, Well_ID, Genotype_Name, JAX_ID, Batch_Info, Cell_Line_Info\

In [27]:
all_automated_plates_combined_fn = "/data/CARDPB2/iNDI/Production/metadata/AbPanel2/AbPanel2_automated_plates_combined.csv"
all_automated_plates_combined = pd.read_csv(all_automated_plates_combined_fn)
all_automated_plates_combined['JAX_ID'] = (
    all_automated_plates_combined['JAX_ID'].str.replace('JAX_ID_', '', regex=False)
)

JAX_catalog_fn = "/data/CARDPB2/iNDI/Production/metadata/JAX_iPSC_Catalog.tsv"
JAX_catalog = pd.read_csv(JAX_catalog_fn, sep='\t')

# clean up all_automated_plates_combined$JAX_ID column to remove 'JAX_ID_" to merge with JAX_catalog$JAX_Code
merged = all_automated_plates_combined.merge(
    JAX_catalog,
    left_on='JAX_ID',
    right_on='JAX_Code',
    how='left',
)
print(merged.head())

                                        Cell_Line_ID iNDI_Plate_ID Well_ID  \
0                 JAX_ID_JIPSC001002\nFUS R495X HOM     INDI00004D     A01   
1  JAX_ID_JIPSC001072\nVCP R159H REV \nLow counts...    INDI00004D     A02   
2              JAX_ID_JIPSC001000\nParental  ID:5424    INDI00004D     A03   
3                 JAX_ID_JIPSC001006\nFUS R495X REV     INDI00004D     A04   
4                  JAX_ID_JIPSC001070\nVCP R159H HET    INDI00004D     A05   

   Genotype_Name       JAX_ID Batch_Info     JAX_Code      Gene Variant  \
0  FUS R495X HOM  JIPSC001002        NaN  JIPSC001002       FUS   R495X   
1  VCP R159H REV  JIPSC001072        NaN  JIPSC001072       VCP   R159H   
2       Parental  JIPSC001000    ID:5424  JIPSC001000  KOLF2.1J     NaN   
3  FUS R495X REV  JIPSC001006        NaN  JIPSC001006       FUS   R495X   
4  VCP R159H HET  JIPSC001070        NaN  JIPSC001070       VCP   R159H   

  Genotype      Gene_Var_Type  
0  SNV/SNV  FUS_R495X_SNV/SNV  
1   REV/WT   VCP

In [28]:
files = glob.glob("/data/kelpschdj/iNDI/Production/outputs/run_20260921_131450_n3r6/nuclei_filtered/*_nuclei_filtered.parquet")

parts = []
for f in files:
    part = pd.read_parquet(f)
    parts.append(part)
    print(f"{os.path.basename(f)}: {len(part)} rows")

df = pd.concat(parts, ignore_index=True)

print(f"\nLoaded {len(files)} files, {len(df)} total rows")
print(df.head())
print(df.columns.tolist())

plate_id_fn = "/data/CARDPB2/iNDI/Production/metadata/indi_plateID_to_folderID.csv"
plate_id = pd.read_csv(plate_id_fn)
print(plate_id.head())

df = df.merge(
    plate_id,
    left_on="Measurement_ID",
    right_on="Folder_ID",
    how="left",
    validate="many_to_one",
)

# Row/Column are 1-based ints from the filename parse (r01c01...).
df["Well_ID"] = (
    df["Row"].astype(int).map(lambda r: string.ascii_uppercase[r - 1])
    + df["Column"].astype(int).map(lambda c: f"{c:02d}")
)

df = df.merge(
    merged[["iNDI_Plate_ID", "Well_ID", "Gene_Var_Type", "Genotype_Name", "Gene", "Variant"]],
    on=["iNDI_Plate_ID", "Well_ID"],
    how="left",
    validate="many_to_one",
)

# sanity check the join
n_unmatched = df["Gene_Var_Type"].isna().sum()
print(f"{n_unmatched} nuclei rows didn't match a genotype")
if n_unmatched:
    print(df.loc[df["Gene_Var_Type"].isna(), ["iNDI_Plate_ID", "Well_ID"]].drop_duplicates().head(20))

print(df.head())

0acd04e9-3bc9-400b-8557-3f5b4123caaa_nuclei_filtered.parquet: 162722 rows
106c8b69-e11f-4698-abd6-d4d01ec5fbb2_nuclei_filtered.parquet: 82775 rows
48fb9a14-1123-488d-9b56-3befe944f933_nuclei_filtered.parquet: 224594 rows
72a1d9f3-277a-4abd-bf55-cbbc07152dff_nuclei_filtered.parquet: 68159 rows
84809677-0531-486c-b79f-e25c7ef9a7c6_nuclei_filtered.parquet: 133722 rows
85cc6732-9e29-40ad-9ec1-e94b9f334ebe_nuclei_filtered.parquet: 87508 rows
9ed6a73a-227d-45a5-90dd-5c87fa906bea_nuclei_filtered.parquet: 171742 rows
a9b2a502-ad2f-4ee9-85ba-10df3cea3b46_nuclei_filtered.parquet: 237108 rows
cba96cb5-1337-4136-ab55-482e1bc9bed3_nuclei_filtered.parquet: 182607 rows
cd5eb985-d862-4c96-8b57-570f462ee2fa_nuclei_filtered.parquet: 268737 rows
d5847ccd-b4ba-4ce9-a214-ea68b817e7f0_nuclei_filtered.parquet: 129787 rows
d5b2e3eb-3b91-4619-8c74-846bbcd92bd4_nuclei_filtered.parquet: 190444 rows
e61d5e8c-faef-4c2c-8df6-6cc72032f19e_nuclei_filtered.parquet: 176028 rows

Loaded 13 files, 2115933 total rows
   l

In [31]:
group_cols = ["iNDI_Plate_ID", "Gene_Var_Type"]

# "Before" = every nucleus in the table; "after" = only selected.
before = (
    df.groupby(group_cols)
      .agg(images_before=("image_name", "nunique"),
           wells_before=("Well_ID", "nunique"),
           nuclei_before=("image_name", "size"))
)

after = (
    df[df["selected"]]
      .groupby(group_cols)
      .agg(images_after=("image_name", "nunique"),
           wells_after=("Well_ID", "nunique"),
           nuclei_after=("image_name", "size"))
)

summary = (
    before.join(after, how="left")
          .fillna({"images_after": 0, "wells_after": 0, "nuclei_after": 0})
          .astype({"images_after": int, "wells_after": int, "nuclei_after": int})
          .reset_index()
)

# reorder for readability
summary = summary[[
    "iNDI_Plate_ID", "Gene_Var_Type",
    "wells_before", "wells_after",
    "images_before", "images_after",
    "nuclei_before", "nuclei_after",
]]

# print grouped by plate
for plate, grp in summary.groupby("iNDI_Plate_ID"):
    print(f"\n=== iNDI_Plate_ID: {plate} ===")
    print(grp.drop(columns="iNDI_Plate_ID").to_string(index=False))

summary.to_csv(
    "/data/kelpschdj/iNDI/Production/outputs/filtering_summary.csv",
    index=False,
)
print("Wrote filtering_summary.csv")


=== iNDI_Plate_ID: INDI00004D ===
    Gene_Var_Type  wells_before  wells_after  images_before  images_after  nuclei_before  nuclei_after
 FUS_R495X_REV/WT            48           48           2850          2815          27215         15721
FUS_R495X_SNV/SNV            48           48           2877          2842          29264         15048
 FUS_R495X_SNV/WT            48           48           2849          2822          33164         17512
         KOLF2.1J            96           96           5797          5747          68711         36908
 VCP_R159H_REV/WT            48           48           2158          2062           8972          6066
VCP_R159H_SNV/SNV            48           48           2917          2906          31516         17624
 VCP_R159H_SNV/WT            47           47           2835          2824          38266         19570

=== iNDI_Plate_ID: INDI00008D ===
       Gene_Var_Type  wells_before  wells_after  images_before  images_after  nuclei_before  nuclei_after
